In [26]:
import datasets as ds
import torch
import tqdm
import peft

from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login

import help_lib as hb


login()

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device

'cuda'

In [3]:
STUDENT_MODEL = "google/gemma-3-1b-it"

seed = 42
dataset_sft_len = 15_000

# Подготовка всех данных

In [4]:
def get_well_distributed_sft_dataset(dataset, ds_len):
    
    ds_3m = dataset.filter(lambda x: len(str(x["target"])) >= 3)
    ds_not_3m = dataset.filter(lambda x: len(str(x["target"])) < 3)
    ds_1 = dataset.filter(lambda x: len(str(x["target"])) == 1)
    
    return ds.concatenate_datasets([ds_1, ds_3m, ds_not_3m.select(range(ds_len - len(ds_1) - len(ds_3m)))]).shuffle(seed=seed)

In [5]:
dataset = ds.load_from_disk("data/dataset")
test_dataset = ds.load_from_disk("data/test_dataset")

raw = get_well_distributed_sft_dataset(dataset, dataset_sft_len)
split = raw.train_test_split(test_size=0.01, seed=42)

Filter: 100%|██████████| 60253/60253 [00:00<00:00, 60568.33 examples/s]


In [6]:
dataset[0]

{'target': 26,
 'nums': [19, 51, 66, 72],
 'messages': [{'role': 'system',
   'content': 'Solve Countdown. Use ALL given numbers ONCE to reach target.Format: <think> steps with = </think> <answer> final equation </answer>. Be concise.'},
  {'role': 'user', 'content': 'Target: 26 | Numbers: [19, 51, 66, 72]'},
  {'role': 'assistant',
   'content': '<think>\n72 - 51 = 21\n47 - 21 = 26\n72 - 66 = 6\n51 - 19 = 32\n</think>\n<answer>(51 - 19) - (72 - 66) = 26</answer>'}]}

In [7]:
raw_train = split["train"]
raw_val = split["test"]

raw_train, raw_val

(Dataset({
     features: ['target', 'nums', 'messages'],
     num_rows: 14850
 }),
 Dataset({
     features: ['target', 'nums', 'messages'],
     num_rows: 150
 }))

In [8]:
raw_train[0]

{'target': 40,
 'nums': [59, 44, 79, 88],
 'messages': [{'role': 'system',
   'content': 'Solve Countdown. Use ALL given numbers ONCE to reach target.Format: <think> steps with = </think> <answer> final equation </answer>. Be concise.'},
  {'role': 'user', 'content': 'Target: 40 | Numbers: [59, 44, 79, 88]'},
  {'role': 'assistant',
   'content': '<think>\n88 - 79 = 9\n59 - 44 = 15\n79 - 59 = 20\n88 - 59 = 29\n79 - 44 = 35\n88 - 44 = 44\n44 - 4 = 40\n(88 - 79) = 9\n(59 - 44) = 15\n15 - 9 = 6\n(79 - 59) = 20\n20 * 2 = 40\n(88 - 44) = 44\n(88 / 44) = 2\n(79 - 59) * (88 / 44) = 40\n88 / 44 = 2\n</think>\n<answer>(79 - 59) * (88 / 44) = 40</answer>'}]}

# SFT этап

In [29]:
def compute_val_acc(
    model, 
    val_dataset,
    tokenizer,
    batch_size, 
    max_new_tokens, 
    device=device
):

    model.eval()
    
    loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=hb.collate_data_fn,
    )

    total = 0
    correct = 0
    for batch in tqdm.tqdm(loader):
        prompts = [
            tokenizer.apply_chat_template(
                [m for m in msgs if m["role"] != "assistant"],
                tokenize=False,
                add_generation_prompt=True
            )
            
            for msgs in batch["messages"]
        ]

        enc = tokenizer(
            prompts,
            padding=True,
            truncation=True,
            return_tensors="pt",
            add_special_tokens=False
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        prompt_lens = enc["attention_mask"].sum(dim=1)

        with torch.inference_mode():
            out = model.generate(
                **enc,
                max_new_tokens=max_new_tokens
            )

        for seq, plen, nums, target in zip(out, prompt_lens, batch["nums"], batch["target"]):
            pred = tokenizer.decode(seq[int(plen):], skip_special_tokens=True).strip()
            total += 1
            
            if "<answer>" in pred:
                pred = hb.extract_last_answer(pred)
            
            pred = hb.extract_generated_equation(pred)
            if hb.is_valid_equation(pred, nums, target):
                correct += 1
    
    
    model.train()
    
    return {
        "solve_acc": correct / max(total, 1),
        "n": total,
    }
    

In [23]:
def get_sft_trainer(train_dataset, model_save_path, lora_r):
    
    model = AutoModelForCausalLM.from_pretrained(STUDENT_MODEL)
    tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
    
    lora_config = peft.LoraConfig(
        r=lora_r,
        lora_alpha=lora_r * 2,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    

    training_args = SFTConfig(
        output_dir=model_save_path,
        
        num_train_epochs=2,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        
        learning_rate=2e-4,
        logging_steps=10,
        save_steps=200,
        lr_scheduler_type="cosine",
        warmup_steps=100,
        
        eval_steps=200,
        max_length=256,
        
        packing=False,
        report_to="tensorboard"
    )
    
    return SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset.select_columns(["messages"]),
        peft_config=lora_config,
        processing_class=tokenizer,
        # formatting_func=format_example
    )


In [ ]:
sft_trainer = get_sft_trainer(raw_train, "./contest/gemma3_SFT_Trainer", lora_r=32)

In [ ]:
sft_trainer.train()

Замерим скор Accuracy на валидационной выборке.
В функции для подсчёта валидации модель делает one-hot предсказания, без Rejection-Sampling

In [32]:
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
acc = compute_val_acc(
    model=sft_trainer.model, 
    val_dataset=raw_val,
    tokenizer=tokenizer,
    batch_size=64, 
    max_new_tokens=256,
    device=device
)

acc


100%|██████████| 3/3 [00:57<00:00, 19.31s/it]


{'solve_acc': 0.5, 'n': 150}

Сохраним модель

In [ ]:
sft_trainer.model.save_pretrained("./contest/gemma3_62")